---

# Sprint 5 - breast-level CC/MLO fusion

Runs after the windowing cells. Uses **only** the frozen wide 0.1-99.9 predictions already
computed on the development set - no image is decoded or scored again, and the windowing
experiment is not repeated.

Question: does combining the CC and MLO views of the same breast rank better than either
view on its own?

**A note on units, stated once and enforced throughout.** Image-level AUC and breast-level
AUC are computed over different populations - 10,558 images against roughly 4,000 breasts.
The difference between them is a change of evaluation unit, not a fusion gain, and it is
never reported as one here. Every comparison below is breast against breast, on exactly the
same breasts.

The sealed RSNA holdout is not touched. No threshold is tuned.

## F1.0 Load the frozen wide-windowing predictions

In [41]:
import os, json, itertools, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

NBOOT_F   = 2000
RNG_F     = np.random.default_rng(90210)
HARM_F    = -0.02      # a subgroup is materially harmed if its paired CI hi sits below this
LOGIT_EPS = 1e-6

# These cells need no GPU and no model bundles - only the saved wide-windowing
# predictions. They run either appended to the windowing notebook, or standalone with
# windowing_paired_scores_full.csv and rsna_split_manifest.csv attached as a dataset.
if 'OUT_DIR' not in globals():
    OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
    os.makedirs(OUT_DIR, exist_ok=True)


def _locate(fname):
    """OUT_DIR first, then anywhere under /kaggle/input, then the working directory."""
    p = os.path.join(OUT_DIR, fname)
    if os.path.exists(p):
        return p
    for root in ('/kaggle/input', '.'):
        if not os.path.isdir(root):
            continue
        for dirpath, _dirs, files in os.walk(root):
            if fname in files:
                return os.path.join(dirpath, fname)
    return None


# --- source the predictions: memory first, then the saved CSV ---
_src = None
if 'wres' in globals() and 'ensemble_wide' in globals()['wres'].columns:
    fus_src = wres.copy()
    _src = 'in-memory wres'
else:
    _p = _locate('windowing_paired_scores_full.csv')
    if _p is None:
        raise SystemExit('windowing_paired_scores_full.csv not found. Either run the '
                         'windowing cells with WIN_SMOKE_N = 0 in this session, or attach '
                         'it as a Kaggle dataset.')
    fus_src = pd.read_csv(_p)
    _src = _p

SCORERS_F = [c[:-5] for c in fus_src.columns if c.endswith('_wide')]
assert SCORERS_F, 'no *_wide columns in the prediction table'
print('source        :', _src)
print('scorers       :', SCORERS_F)
print('images        : %d   patients %d   cancers %d'
      % (len(fus_src), fus_src.patient_id.nunique(), int(fus_src.cancer.sum())))

# --- guard: these must be development patients only ---
_mp = _locate('rsna_split_manifest.csv')
if _mp:
    _man = pd.read_csv(_mp)
    _dev = set(_man.loc[_man.split == 'dev', 'patient_id'])
    _hold = set(_man.loc[_man.split == 'holdout', 'patient_id'])
    _leak = set(fus_src.patient_id) & _hold
    if _leak:
        raise SystemExit('HOLDOUT LEAK: %d holdout patients are in the prediction table. Stop.'
                         % len(_leak))
    print('holdout leak  : none  [PASS]  (%d dev patients on record)' % len(_dev))
else:
    print('WARNING: split manifest not found, cannot verify the holdout is excluded.')

# --- frozen windowing, for the record ---
_fp = _locate('frozen_preprocessing.json')
if _fp:
    print('frozen window :', json.load(open(_fp)).get('window_percentiles'))

source        : in-memory wres
scorers       : ['v11_dro', 'v8_resnet', 'ensemble']
images        : 10558   patients 6214   cancers 800
holdout leak  : none  [PASS]  (6214 dev patients on record)
frozen window : [0.1, 99.9]


## F1.1 Data audit

The breast identifier is `patient_id` + `laterality`, so left and right are separate units
and can never be pooled. Before any fusion, four things are checked and counted:

- **label consistency** - every image of a breast must carry the same `cancer` value
- **view coverage** - how many breasts have both views, CC only, MLO only, or neither
- **duplicate views** - breasts with more than one image of the same view
- **unusable rows** - views that are neither CC nor MLO

A breast with an inconsistent label is dropped rather than guessed at, and the count is
reported.

In [42]:
fus = fus_src.copy()
fus['view'] = fus['view'].astype(str).str.upper().str.strip()
fus['laterality'] = fus['laterality'].astype(str).str.upper().str.strip()
fus['breast'] = fus.patient_id.astype(str) + '_' + fus.laterality

_bad_view = fus[~fus.view.isin(['CC', 'MLO'])]
print('rows with a view other than CC/MLO: %d' % len(_bad_view))
if len(_bad_view):
    print(_bad_view.view.value_counts().to_string())
fus = fus[fus.view.isin(['CC', 'MLO'])].copy()

_bad_lat = fus[~fus.laterality.isin(['L', 'R'])]
print('rows with a laterality other than L/R: %d' % len(_bad_lat))
fus = fus[fus.laterality.isin(['L', 'R'])].copy()

# --- label consistency within a breast ---
lab = fus.groupby('breast').cancer.agg(['nunique', 'max', 'size'])
inconsistent = lab[lab['nunique'] > 1]
print('breasts with inconsistent cancer labels: %d' % len(inconsistent))
if len(inconsistent):
    print(inconsistent.head(10).to_string())
    fus = fus[~fus.breast.isin(inconsistent.index)].copy()
    print('  -> dropped')

# --- view coverage and duplicates ---
cnt = (fus.groupby(['breast', 'view']).size().unstack(fill_value=0)
       .reindex(columns=['CC', 'MLO'], fill_value=0))
cov = pd.DataFrame({
    'both views': [int(((cnt.CC > 0) & (cnt.MLO > 0)).sum())],
    'CC only': [int(((cnt.CC > 0) & (cnt.MLO == 0)).sum())],
    'MLO only': [int(((cnt.CC == 0) & (cnt.MLO > 0)).sum())],
    'dup CC (>1 image)': [int((cnt.CC > 1).sum())],
    'dup MLO (>1 image)': [int((cnt.MLO > 1).sum())],
}).T.rename(columns={0: 'breasts'})
print()
print(cov.to_string())

audit = cov.copy()
audit.loc['rows dropped, bad view'] = len(_bad_view)
audit.loc['rows dropped, bad laterality'] = len(_bad_lat)
audit.loc['breasts dropped, inconsistent label'] = len(inconsistent)
audit.loc['breasts total'] = fus.breast.nunique()
audit.loc['patients total'] = fus.patient_id.nunique()
audit.index.name = 'item'
audit.to_csv(os.path.join(OUT_DIR, 'fusion_data_audit.csv'))
print()
print('saved fusion_data_audit.csv')

rows with a view other than CC/MLO: 10
view
LM    4
AT    3
ML    3
rows with a laterality other than L/R: 0
breasts with inconsistent cancer labels: 0

                    breasts
both views             1705
CC only                3286
MLO only               3364
dup CC (>1 image)       194
dup MLO (>1 image)      280

saved fusion_data_audit.csv


## F1.2 Build the breast table

Two steps, in this order:

1. **Within a view**, a breast with more than one image gets the *mean* of those
   predictions. A duplicate view is repeat imaging of the same anatomy, so averaging is
   the neutral choice - taking the maximum here would smuggle a second, hidden fusion rule
   into the "CC only" baseline and make the comparison unfair.
2. **Across views**, the fusion rules below are applied.

The primary cohort is breasts holding **both** views. Single-view breasts are counted and
set aside, because a fusion method cannot be compared against itself on them.

In [43]:
val_cols = [s + '_wide' for s in SCORERS_F]

# step 1: mean within view
byview = (fus.groupby(['patient_id', 'breast', 'laterality', 'view'], as_index=False)
             .agg(cancer=('cancer', 'max'),
                  n_img=('image_id', 'size'),
                  **{c: (c, 'mean') for c in val_cols}))

# carry one set of covariates per breast (mode, so a disagreement resolves to the common value)
def _first(s):
    m = s.mode()
    return m.iat[0] if len(m) else np.nan


cov_cols = [c for c in ['site_id', 'machine_id', 'density'] if c in fus.columns]
brcov = fus.groupby('breast').agg(**{c: (c, _first) for c in cov_cols},
                                  age=('age', 'median'))
brcov['age_band'] = pd.cut(brcov.age, [0, 50, 60, 70, 200],
                           labels=['<50', '50-60', '60-70', '70+'], right=False)

wide = byview.pivot_table(index=['patient_id', 'breast', 'laterality'],
                          columns='view', values=val_cols + ['cancer'], aggfunc='first')
wide.columns = ['%s__%s' % (a, b) for a, b in wide.columns]
wide = wide.reset_index()

wide['cancer'] = wide[[c for c in wide.columns if c.startswith('cancer__')]].max(axis=1)
has_cc = wide[[c for c in wide.columns if c.endswith('__CC')]].notna().any(axis=1)
has_ml = wide[[c for c in wide.columns if c.endswith('__MLO')]].notna().any(axis=1)

BR = wide[has_cc & has_ml].merge(brcov, on='breast', how='left').reset_index(drop=True)

print('breasts with both views (primary cohort): %d   patients %d   cancer breasts %d'
      % (len(BR), BR.patient_id.nunique(), int(BR.cancer.sum())))
print('set aside, single view                  : %d' % int((~(has_cc & has_ml)).sum()))
print()
print('primary cohort, cancer breasts by stratum:')
for c in ['site_id', 'density', 'age_band', 'machine_id']:
    if c in BR.columns:
        t = BR.groupby(c, observed=True).cancer.agg(['size', 'sum'])
        t.columns = ['breasts', 'cancer breasts']
        print('\n%s\n%s' % (c, t.to_string()))

breasts with both views (primary cohort): 1705   patients 1624   cancer breasts 343
set aside, single view                  : 6650

primary cohort, cancer breasts by stratum:

site_id
         breasts  cancer breasts
site_id                         
1           1049           178.0
2            656           165.0

density
         breasts  cancer breasts
density                         
A            120            12.0
B            445            80.0
C            432            82.0
D             50             4.0

age_band
          breasts  cancer breasts
age_band                         
<50           331            30.0
50-60         547            82.0
60-70         540           131.0
70+           285           100.0

machine_id
            breasts  cancer breasts
machine_id                         
21              208            52.0
29              211            55.0
48              237            58.0
49              867           165.0
93               56             4.0

## F1.3 The five methods

| method | definition |
|---|---|
| `cc_only` | the CC probability |
| `mlo_only` | the MLO probability |
| `mean_prob` | (CC + MLO) / 2 |
| `max_prob` | max(CC, MLO) - matches the engine's highest-severity rule |
| `mean_logit` | sigmoid of the mean of the two logits, probabilities clipped to 1e-6 |

`mean_logit` differs from `mean_prob` because averaging in logit space weights confident
predictions more heavily; on well-separated scores the two often agree closely, and where
they diverge it is worth knowing which way.

**Logistic-regression fusion is not fitted here.** Its weights would have to come from a
separate Mammo-Bench internal validation set; fitting them on RSNA development data would
mean the fusion rule and the evaluation share a dataset. The cell checks for such a file
and skips with a stated reason if it is absent.

In [44]:
def _logit(p):
    p = np.clip(np.asarray(p, dtype=np.float64), LOGIT_EPS, 1 - LOGIT_EPS)
    return np.log(p / (1 - p))


def _sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


METHODS = ['cc_only', 'mlo_only', 'mean_prob', 'max_prob', 'mean_logit']

for s in SCORERS_F:
    cc = BR['%s_wide__CC' % s].to_numpy(dtype=np.float64)
    ml = BR['%s_wide__MLO' % s].to_numpy(dtype=np.float64)
    BR['%s|cc_only' % s] = cc
    BR['%s|mlo_only' % s] = ml
    BR['%s|mean_prob' % s] = (cc + ml) / 2.0
    BR['%s|max_prob' % s] = np.maximum(cc, ml)
    BR['%s|mean_logit' % s] = _sigmoid((_logit(cc) + _logit(ml)) / 2.0)

COLS = ['%s|%s' % (s, m) for s in SCORERS_F for m in METHODS]
assert BR[COLS].notna().all().all(), 'NaNs in the fused columns - investigate before reading on'
print('built %d method columns over %d breasts' % (len(COLS), len(BR)))
print(BR[COLS].describe().T[['mean', 'std', 'min', 'max']].round(4).to_string())

# --- logistic-regression fusion: only from a clean internal validation set ---
_lr_path = os.path.join(OUT_DIR, 'mammobench_val_view_probs.csv')
if os.path.exists(_lr_path):
    print('\nInternal validation predictions found at %s.' % _lr_path)
    print('Logistic fusion can be fitted from that file; not implemented in this pass.')
else:
    print('\nLogistic-regression fusion SKIPPED: no Mammo-Bench internal validation '
          'predictions available. Fitting weights on RSNA development data would put the '
          'fusion rule and the evaluation on the same dataset.')

built 15 method columns over 1705 breasts
                        mean     std     min     max
v11_dro|cc_only       0.6357  0.1869  0.0143  0.9967
v11_dro|mlo_only      0.6199  0.1720  0.0331  0.9954
v11_dro|mean_prob     0.6278  0.1557  0.0420  0.9955
v11_dro|max_prob      0.6961  0.1571  0.0509  0.9967
v11_dro|mean_logit    0.6324  0.1612  0.0411  0.9955
v8_resnet|cc_only     0.5272  0.2719  0.0071  0.9923
v8_resnet|mlo_only    0.4516  0.2583  0.0040  0.9889
v8_resnet|mean_prob   0.4894  0.2412  0.0178  0.9868
v8_resnet|max_prob    0.5766  0.2654  0.0230  0.9923
v8_resnet|mean_logit  0.4913  0.2501  0.0168  0.9868
ensemble|cc_only      0.6420  0.1990  0.0197  0.9930
ensemble|mlo_only     0.5959  0.1848  0.0386  0.9937
ensemble|mean_prob    0.6190  0.1741  0.0582  0.9923
ensemble|max_prob     0.6844  0.1778  0.0777  0.9937
ensemble|mean_logit   0.6232  0.1787  0.0550  0.9925

Logistic-regression fusion SKIPPED: no Mammo-Bench internal validation predictions available. Fitting weights

## F1.4 Shared bootstrap

Patients are resampled once, and **every method and every model is evaluated on the same
resamples**. That is what makes the differences paired: the interval is on the difference
between two methods on identical breasts, not the overlap of two independent intervals.

Resamples in which a stratum ends up all-positive or all-negative are dropped for every
column at once, so the columns stay aligned.

In [45]:
def make_boot(df, nboot=NBOOT_F, rng=None):
    rng = rng or RNG_F
    pats = df.patient_id.to_numpy()
    y = df.cancer.to_numpy(dtype=int)
    uniq = np.unique(pats)
    idx = {p: np.where(pats == p)[0] for p in uniq}
    boots = []
    for _ in range(nboot):
        rows = np.concatenate([idx[p] for p in rng.choice(uniq, len(uniq), replace=True)])
        yy = y[rows]
        if yy.sum() == 0 or yy.sum() == len(yy):
            continue
        boots.append(rows.astype(np.int32))
    return boots


def boot_auc_matrix(df, cols, boots):
    y = df.cancer.to_numpy(dtype=int)
    M = np.empty((len(boots), len(cols)), dtype=np.float64)
    V = {c: df[c].to_numpy(dtype=np.float64) for c in cols}
    for b, rows in enumerate(boots):
        yy = y[rows]
        for j, c in enumerate(cols):
            M[b, j] = roc_auc_score(yy, V[c][rows])
    return M


BOOTS = make_boot(BR)
print('bootstrap resamples kept: %d of %d   unit: patient_id' % (len(BOOTS), NBOOT_F))
BM = boot_auc_matrix(BR, COLS, BOOTS)
BOOTDF = pd.DataFrame(BM, columns=COLS)
BOOTDF.to_csv(os.path.join(OUT_DIR, 'fusion_bootstrap.csv'), index=False)
print('saved fusion_bootstrap.csv  (%d x %d)' % BOOTDF.shape)

bootstrap resamples kept: 2000 of 2000   unit: patient_id
saved fusion_bootstrap.csv  (2000 x 15)


## F1.5 Breast-level AUC, and every fusion method against both single views

In [46]:
y_br = BR.cancer.to_numpy(dtype=int)
npos, nneg = int(y_br.sum()), int((1 - y_br).sum())

rows = []
for s in SCORERS_F:
    pt = {m: roc_auc_score(y_br, BR['%s|%s' % (s, m)]) for m in METHODS}
    strong = 'cc_only' if pt['cc_only'] >= pt['mlo_only'] else 'mlo_only'
    for m in METHODS:
        col = '%s|%s' % (s, m)
        lo, hi = np.percentile(BOOTDF[col], [2.5, 97.5])
        r = dict(scorer=s, method=m, n_breasts=len(BR), pos=npos, neg=nneg,
                 auc=pt[m], auc_lo=lo, auc_hi=hi)
        for ref in ['cc_only', 'mlo_only']:
            d = BOOTDF[col] - BOOTDF['%s|%s' % (s, ref)]
            r['vs_%s' % ref] = pt[m] - pt[ref]
            r['vs_%s_lo' % ref] = np.percentile(d, 2.5)
            r['vs_%s_hi' % ref] = np.percentile(d, 97.5)
        d = BOOTDF[col] - BOOTDF['%s|%s' % (s, strong)]
        r['stronger_single'] = strong
        r['vs_stronger'] = pt[m] - pt[strong]
        r['vs_stronger_lo'] = np.percentile(d, 2.5)
        r['vs_stronger_hi'] = np.percentile(d, 97.5)
        rows.append(r)

summary = pd.DataFrame(rows)
summary.to_csv(os.path.join(OUT_DIR, 'fusion_summary.csv'), index=False)

for s in SCORERS_F:
    print('=== %s ===   %d breasts, %d cancer, %d non-cancer' % (s, len(BR), npos, nneg))
    v = summary[summary.scorer == s]
    print(v[['method', 'auc', 'auc_lo', 'auc_hi', 'vs_cc_only', 'vs_cc_only_lo',
             'vs_cc_only_hi', 'vs_mlo_only', 'vs_mlo_only_lo', 'vs_mlo_only_hi']]
          .round(4).to_string(index=False))
    print()
print('saved fusion_summary.csv')
print('All figures are breast-level on the same %d breasts. Image-level AUC is a different '
      'evaluation unit and is not comparable to any number above.' % len(BR))

=== v11_dro ===   1705 breasts, 343 cancer, 1362 non-cancer
    method    auc  auc_lo  auc_hi  vs_cc_only  vs_cc_only_lo  vs_cc_only_hi  vs_mlo_only  vs_mlo_only_lo  vs_mlo_only_hi
   cc_only 0.6725  0.6376  0.7062      0.0000         0.0000         0.0000       0.0307         -0.0056          0.0631
  mlo_only 0.6419  0.6088  0.6749     -0.0307        -0.0631         0.0056       0.0000          0.0000          0.0000
 mean_prob 0.6777  0.6451  0.7092      0.0052        -0.0119         0.0238       0.0359          0.0166          0.0532
  max_prob 0.6863  0.6514  0.7180      0.0138        -0.0045         0.0349       0.0445          0.0211          0.0675
mean_logit 0.6830  0.6498  0.7145      0.0104        -0.0062         0.0291       0.0411          0.0209          0.0598

=== v8_resnet ===   1705 breasts, 343 cancer, 1362 non-cancer
    method    auc  auc_lo  auc_hi  vs_cc_only  vs_cc_only_lo  vs_cc_only_hi  vs_mlo_only  vs_mlo_only_lo  vs_mlo_only_hi
   cc_only 0.5516  0.5164  0.5

## F1.6 Subgroups

Site, density, age band and machine. Each stratum gets its own patient bootstrap, shared
across methods within that stratum so the differences stay paired. Strata with fewer than
ten positive or ten negative breasts return no estimate rather than an unstable one.

In [47]:
MIN_POS = MIN_NEG = 10


def strata_of(df):
    out = []
    for c in ['site_id', 'density', 'age_band', 'machine_id']:
        if c not in df.columns:
            continue
        for k, g in df.dropna(subset=[c]).groupby(c, observed=True):
            out.append(('%s %s' % (c.replace('_id', ''), k), g))
    return out


sub_rows = []
for label, g in strata_of(BR):
    ypos, yneg = int(g.cancer.sum()), int((1 - g.cancer).sum())
    if ypos < MIN_POS or yneg < MIN_NEG:
        sub_rows.append(dict(stratum=label, scorer='-', method='-', n_breasts=len(g),
                             pos=ypos, neg=yneg, auc=np.nan, vs_stronger=np.nan,
                             lo=np.nan, hi=np.nan, note='too few of one class'))
        continue
    gb = make_boot(g, rng=np.random.default_rng(abs(hash(label)) % (2 ** 31)))
    if not gb:
        continue
    gm = pd.DataFrame(boot_auc_matrix(g, COLS, gb), columns=COLS)
    yy = g.cancer.to_numpy(dtype=int)
    for s in SCORERS_F:
        pt = {m: roc_auc_score(yy, g['%s|%s' % (s, m)]) for m in METHODS}
        strong = 'cc_only' if pt['cc_only'] >= pt['mlo_only'] else 'mlo_only'
        for m in METHODS:
            d = gm['%s|%s' % (s, m)] - gm['%s|%s' % (s, strong)]
            sub_rows.append(dict(stratum=label, scorer=s, method=m, n_breasts=len(g),
                                 pos=ypos, neg=yneg, auc=pt[m],
                                 stronger_single=strong, vs_stronger=pt[m] - pt[strong],
                                 lo=np.percentile(d, 2.5), hi=np.percentile(d, 97.5),
                                 note=''))

subs = pd.DataFrame(sub_rows)
subs.to_csv(os.path.join(OUT_DIR, 'fusion_subgroups.csv'), index=False)

_ens = SCORERS_F[-1] if 'ensemble' not in SCORERS_F else 'ensemble'
print('=== %s, fusion methods against the stronger single view, by stratum ===' % _ens)
v = subs[(subs.scorer.isin([_ens, '-'])) & (subs.method.isin(['-', 'mean_prob', 'max_prob',
                                                              'mean_logit']))]
print(v[['stratum', 'method', 'n_breasts', 'pos', 'neg', 'auc', 'vs_stronger',
         'lo', 'hi', 'note']].round(4).to_string(index=False))
print()
print('saved fusion_subgroups.csv')

=== ensemble, fusion methods against the stronger single view, by stratum ===
       stratum     method  n_breasts  pos  neg    auc  vs_stronger      lo      hi                 note
        site 1  mean_prob       1049  178  871 0.6270      -0.0018 -0.0286  0.0249                     
        site 1   max_prob       1049  178  871 0.6325       0.0037 -0.0165  0.0248                     
        site 1 mean_logit       1049  178  871 0.6291       0.0003 -0.0246  0.0261                     
        site 2  mean_prob        656  165  491 0.6133      -0.0087 -0.0273  0.0089                     
        site 2   max_prob        656  165  491 0.6248       0.0029 -0.0151  0.0228                     
        site 2 mean_logit        656  165  491 0.6157      -0.0063 -0.0238  0.0106                     
     density A  mean_prob        120   12  108 0.6111      -0.0208 -0.1466  0.1001                     
     density A   max_prob        120   12  108 0.6096      -0.0224 -0.1199  0.0844        

## F1.7 Decision

A fusion method is adopted only if **all** of these hold for the ensemble:

1. its AUC exceeds **both** `cc_only` and `mlo_only`
2. the paired 95% interval against the **stronger** single view excludes zero
3. neither individual model is significantly worse than its own stronger single view
4. no sufficiently sized subgroup is materially harmed - no stratum with at least 10
   positive and 10 negative breasts whose paired interval sits entirely below −0.02

If more than one method passes, the highest ensemble AUC wins. If none passes, the stronger
single view is kept and the record states that fusion was **not supported**, which is a
result rather than a failure.

In [48]:
ENS_NAME = 'ensemble' if 'ensemble' in SCORERS_F else SCORERS_F[-1]
es = summary[summary.scorer == ENS_NAME].set_index('method')
strong_ens = es.loc['mean_prob', 'stronger_single']

print('ensemble stronger single view: %s (AUC %.4f)'
      % (strong_ens, es.loc[strong_ens, 'auc']))
print()

candidates = {}
for m in ['mean_prob', 'max_prob', 'mean_logit']:
    c1 = bool(es.loc[m, 'auc'] > es.loc['cc_only', 'auc'] and
              es.loc[m, 'auc'] > es.loc['mlo_only', 'auc'])
    c2 = bool(es.loc[m, 'vs_stronger_lo'] > 0)

    c3, model_notes = True, []
    for s in SCORERS_F:
        if s == ENS_NAME:
            continue
        r = summary[(summary.scorer == s) & (summary.method == m)].iloc[0]
        worse = bool(r['vs_stronger_hi'] < 0)
        model_notes.append('%s %+.4f [%+.4f, %+.4f]%s'
                           % (s, r['vs_stronger'], r['vs_stronger_lo'],
                              r['vs_stronger_hi'], ' LOSS' if worse else ''))
        if worse:
            c3 = False

    harmed = subs[(subs.scorer == ENS_NAME) & (subs.method == m) &
                  (subs.pos >= MIN_POS) & (subs.neg >= MIN_NEG) & (subs.hi < HARM_F)]
    c4 = bool(len(harmed) == 0)

    ok = c1 and c2 and c3 and c4
    candidates[m] = dict(auc=float(es.loc[m, 'auc']),
                         vs_stronger=float(es.loc[m, 'vs_stronger']),
                         lo=float(es.loc[m, 'vs_stronger_lo']),
                         hi=float(es.loc[m, 'vs_stronger_hi']),
                         c1=c1, c2=c2, c3=c3, c4=c4, passes=ok,
                         model_notes=model_notes, harmed=harmed.stratum.tolist())

    print('--- %s : AUC %.4f, vs %s %+.4f [%+.4f, %+.4f]'
          % (m, es.loc[m, 'auc'], strong_ens, es.loc[m, 'vs_stronger'],
             es.loc[m, 'vs_stronger_lo'], es.loc[m, 'vs_stronger_hi']))
    print('    1 beats both single views      :', 'PASS' if c1 else 'FAIL')
    print('    2 paired CI excludes zero      :', 'PASS' if c2 else 'FAIL')
    print('    3 no individual model loss     :', 'PASS' if c3 else 'FAIL',
          ' | ' + '; '.join(model_notes))
    print('    4 no material subgroup harm    :', 'PASS' if c4 else 'FAIL',
          '' if c4 else ' | ' + ', '.join(harmed.stratum.tolist()))
    print('    ->', 'PASSES' if ok else 'rejected')
    print()

passing = {m: v for m, v in candidates.items() if v['passes']}
if passing:
    CHOSEN = max(passing, key=lambda m: passing[m]['auc'])
    supported = True
else:
    CHOSEN = strong_ens
    supported = False

print('=' * 70)
if supported:
    print('DECISION: ADOPT %s' % CHOSEN.upper())
    print('  ensemble breast AUC %.4f against %.4f for %s, %+.4f [%+.4f, %+.4f]'
          % (es.loc[CHOSEN, 'auc'], es.loc[strong_ens, 'auc'], strong_ens,
             es.loc[CHOSEN, 'vs_stronger'], es.loc[CHOSEN, 'vs_stronger_lo'],
             es.loc[CHOSEN, 'vs_stronger_hi']))
else:
    print('DECISION: FUSION NOT SUPPORTED - keep %s' % strong_ens.upper())
    print('  no method beat the stronger single view with an interval excluding zero.')
print('FUSION RULE IS NOW FROZEN')
print('=' * 70)

ensemble stronger single view: cc_only (AUC 0.6030)

--- mean_prob : AUC 0.5952, vs cc_only -0.0078 [-0.0235, +0.0087]
    1 beats both single views      : FAIL
    2 paired CI excludes zero      : FAIL
    3 no individual model loss     : PASS  | v11_dro +0.0052 [-0.0119, +0.0238]; v8_resnet -0.0055 [-0.0215, +0.0106]
    4 no material subgroup harm    : PASS 
    -> rejected

--- max_prob : AUC 0.6050, vs cc_only +0.0020 [-0.0111, +0.0158]
    1 beats both single views      : PASS
    2 paired CI excludes zero      : FAIL
    3 no individual model loss     : PASS  | v11_dro +0.0138 [-0.0045, +0.0349]; v8_resnet -0.0037 [-0.0159, +0.0089]
    4 no material subgroup harm    : PASS 
    -> rejected

--- mean_logit : AUC 0.5975, vs cc_only -0.0055 [-0.0204, +0.0103]
    1 beats both single views      : FAIL
    2 paired CI excludes zero      : FAIL
    3 no individual model loss     : PASS  | v11_dro +0.0104 [-0.0062, +0.0291]; v8_resnet -0.0073 [-0.0235, +0.0083]
    4 no material subgr

### F1.8 Freeze it and write the report

In [49]:
frozen_fusion = {
    'decision': CHOSEN,
    'fusion_supported': bool(supported),
    'breast_identifier': 'patient_id + "_" + laterality; left and right never pooled',
    'within_view_duplicates': 'mean of predictions of the same view',
    'across_view_rule': CHOSEN,
    'logit_clip': LOGIT_EPS,
    'primary_cohort': 'breasts holding both CC and MLO',
    'n_breasts': int(len(BR)),
    'n_patients': int(BR.patient_id.nunique()),
    'n_cancer_breasts': int(BR.cancer.sum()),
    'evaluated_on': 'RSNA development split only; holdout sealed',
    'windowing': 'wide 0.1-99.9, frozen',
    'ensemble_auc': {m: float(es.loc[m, 'auc']) for m in METHODS},
    'stronger_single_view': strong_ens,
    'bootstrap': {'resamples': len(BOOTS), 'unit': 'patient_id', 'shared_across_methods': True},
    'harm_threshold': HARM_F,
    'logistic_fusion': 'skipped - no clean Mammo-Bench internal validation set',
    'criteria': {m: {k: v[k] for k in ('c1', 'c2', 'c3', 'c4', 'passes', 'vs_stronger',
                                       'lo', 'hi')} for m, v in candidates.items()},
}
with open(os.path.join(OUT_DIR, 'frozen_fusion.json'), 'w') as f:
    json.dump(frozen_fusion, f, indent=2, default=float)

L = []
L.append('# Sprint 5 - breast-level CC/MLO fusion decision')
L.append('')
L.append('**Decision: %s**' % ('adopt `%s`' % CHOSEN if supported
                               else 'fusion not supported, keep `%s`' % strong_ens))
L.append('')
L.append('## Cohort')
L.append('')
L.append('- %d breasts holding both CC and MLO, from %d patients, %d cancer breasts'
         % (len(BR), BR.patient_id.nunique(), int(BR.cancer.sum())))
L.append('- RSNA development split only. The holdout was not accessed.')
L.append('- Predictions are the frozen wide 0.1-99.9 windowing pass. Nothing was re-scored.')
L.append('- Duplicate views within a breast were averaged before fusion, so no hidden '
         'second fusion rule enters the single-view baselines.')
L.append('')
L.append('## Ensemble, breast level')
L.append('')
L.append('| method | AUC | 95% CI | vs ' + strong_ens + ' | 95% CI |')
L.append('|---|---|---|---|---|')
for m in METHODS:
    L.append('| `%s` | %.4f | %.4f to %.4f | %+.4f | %+.4f to %+.4f |'
             % (m, es.loc[m, 'auc'], es.loc[m, 'auc_lo'], es.loc[m, 'auc_hi'],
                es.loc[m, 'vs_stronger'], es.loc[m, 'vs_stronger_lo'],
                es.loc[m, 'vs_stronger_hi']))
L.append('')
L.append('## Criteria')
L.append('')
for m, v in candidates.items():
    L.append('- **`%s`** - beats both single views: %s; paired CI excludes zero: %s; '
             'no individual model loss: %s; no material subgroup harm: %s -> **%s**'
             % (m, 'yes' if v['c1'] else 'no', 'yes' if v['c2'] else 'no',
                'yes' if v['c3'] else 'no', 'yes' if v['c4'] else 'no',
                'passes' if v['passes'] else 'rejected'))
    L.append('  - per model against its own stronger single view: %s' % '; '.join(v['model_notes']))
    if v['harmed']:
        L.append('  - harmed strata: %s' % ', '.join(v['harmed']))
L.append('')
L.append('## Notes on interpretation')
L.append('')
L.append('- Every figure here is **breast-level**. Image-level AUC is computed over a '
         'different population and the difference between the two is a change of '
         'evaluation unit, not a fusion gain.')
L.append('- Logistic-regression fusion was skipped: its weights would have to be fitted on '
         'a Mammo-Bench internal validation set, and none is available in this environment. '
         'Fitting them on RSNA development data would put the fusion rule and the '
         'evaluation on the same dataset.')
L.append('- No threshold was tuned. Calibration is the next task.')
L.append('')
_rep = os.path.join(OUT_DIR, 'fusion_decision.md')
with open(_rep, 'w') as f:
    f.write('\n'.join(L))

print('saved frozen_fusion.json')
print('saved fusion_decision.md')
print()
print('\n'.join(L[:24]))

saved frozen_fusion.json
saved fusion_decision.md

# Sprint 5 - breast-level CC/MLO fusion decision

**Decision: fusion not supported, keep `cc_only`**

## Cohort

- 1705 breasts holding both CC and MLO, from 1624 patients, 343 cancer breasts
- RSNA development split only. The holdout was not accessed.
- Predictions are the frozen wide 0.1-99.9 windowing pass. Nothing was re-scored.
- Duplicate views within a breast were averaged before fusion, so no hidden second fusion rule enters the single-view baselines.

## Ensemble, breast level

| method | AUC | 95% CI | vs cc_only | 95% CI |
|---|---|---|---|---|
| `cc_only` | 0.6030 | 0.5675 to 0.6373 | +0.0000 | +0.0000 to +0.0000 |
| `mlo_only` | 0.5749 | 0.5407 to 0.6099 | -0.0281 | -0.0579 to +0.0029 |
| `mean_prob` | 0.5952 | 0.5609 to 0.6289 | -0.0078 | -0.0235 to +0.0087 |
| `max_prob` | 0.6050 | 0.5700 to 0.6397 | +0.0020 | -0.0111 to +0.0158 |
| `mean_logit` | 0.5975 | 0.5638 to 0.6312 | -0.0055 | -0.0204 to +0.0103 |

## Criteria

-

## F1.9 What was written

| file | contents |
|---|---|
| `fusion_data_audit.csv` | view coverage, duplicates, dropped rows, inconsistent labels |
| `fusion_summary.csv` | AUC and paired deltas for every method x model |
| `fusion_bootstrap.csv` | the raw bootstrap AUC matrix, shared resamples |
| `fusion_subgroups.csv` | site, density, age, machine, with breast counts |
| `frozen_fusion.json` | the frozen rule and every criterion that produced it |
| `fusion_decision.md` | the short written decision |

Download these from `/kaggle/working` before the session ends.

**Stop here.** Threshold calibration is the next task, and it must not be started on the
same pass - the fusion rule has to be frozen first, or the threshold gets tuned against a
moving target. The holdout stays sealed.